# Optimización de Tiempos de Atención en Urgencias mediante Modelos Predictivos de Machine Learning

### ¡SIGU - Sistema Inteligente de Gestion de Urgencias!

Se descargó el dataset en un csv para poder hacer el experimento reproducible del sitio oficial de datos abiertos

Informacion del Dataset: Clasificación en Triage Urgencias Salud y Protección Social\
Registros de usuarios que fueron atendidos y clasificados en triage en los servicios de urgencias de las Instituciones prestadoras de servicios de salud de la Empresa Social del Estado Pasto Salud. Es un indicador de Calidad de la resolución 256

Enlace: https://www.datos.gov.co/Salud-y-Protecci-n-Social/Clasificaci-n-en-Triage-Urgencias/vt5n-eu2r/about_data





In [49]:
import pandas as pd

In [50]:
data = pd.read_csv('C:\\Users\\soporte.sistemas\\RepoLocal\\Unicafam\\Project_Analitica_II\\data\\Clasificación_en_Triage_Urgencias_20261003.csv', encoding='latin-1')
data.head() # Mostramos las primeras filas del dataset

,CodAdminis,Nom_Admini,Fecha_Ing,Hora_Ingre,Fecha_Atencion,Hora_Atencion,Triage,Ips,Red
0,ESS118,EMSSANAR S.A.S,2020 Jan 23 08:29:02 AM,1900 Jan 01 08:29:00 AM,2020 Jan 23 08:36:00 AM,1900 Jan 01 08:36:00 AM,III,HOSPITAL LOCAL CIVIL,RED NORTE
1,ESS118,EMSSANAR S.A.S,2020 Jan 23 08:29:02 AM,1900 Jan 01 08:29:00 AM,2020 Jan 23 08:36:00 AM,1900 Jan 01 08:36:00 AM,III,HOSPITAL LOCAL CIVIL,RED NORTE
2,ESS118,EMSSANAR S.A.S,2020 Jan 14 09:27:46 AM,1900 Jan 01 09:27:00 AM,2020 Jan 14 10:18:00 AM,1900 Jan 01 10:18:00 AM,III,HOSPITAL LOCAL CIVIL,RED NORTE
3,ESS118,EMSSANAR S.A.S,2020 Feb 29 09:01:31 AM,1900 Jan 01 09:01:00 AM,2020 Feb 29 09:11:00 AM,1900 Jan 01 09:11:00 AM,III,CENTRO HOSPITAL LA ROSA,RED SUR
4,ESS118,EMSSANAR S.A.S,2020 Feb 09 05:26:00 PM,1900 Jan 01 05:26:00 PM,2020 Feb 09 05:50:00 PM,1900 Jan 01 05:50:00 PM,III,HOSPITAL LOCAL CIVIL,RED NORTE


In [51]:
data.shape # Validamos las dimensiones del dataset

(89453, 9)

In [52]:
data.info() # Información general del dataset

<class 'pandas.DataFrame'>
RangeIndex: 89453 entries, 0 to 89452
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   CodAdminis      89452 non-null  str  
 1   Nom_Admini      89452 non-null  str  
 2   Fecha_Ing       89453 non-null  str  
 3   Hora_Ingre      89453 non-null  str  
 4   Fecha_Atencion  89453 non-null  str  
 5   Hora_Atencion   89453 non-null  str  
 6   Triage          89453 non-null  str  
 7   Ips             89453 non-null  str  
 8   Red             89453 non-null  str  
dtypes: str(9)
memory usage: 6.1 MB


In [53]:
# Verificamos si hay filas duplicadas en el dataset

data.duplicated().sum() 

np.int64(9932)

In [54]:
# Eliminamos las filas duplicadas del dataset

data = data.drop_duplicates().copy()
data.duplicated().sum()  # Verificamos nuevamente si hay filas duplicadas

np.int64(0)

In [55]:
data.shape # Validamos las dimensiones del dataset después de eliminar duplicados

(79521, 9)

In [56]:
# Convertir fechas completas (ignorando Hora_Ingre y Hora_Atencion con año 1900)

formato_fecha = "%Y %b %d %I:%M:%S %p"

data['dt_ingreso'] = pd.to_datetime(data['Fecha_Ing'], format=formato_fecha, errors='coerce')
data['dt_atencion'] = pd.to_datetime(data['Fecha_Atencion'], format=formato_fecha, errors='coerce')

In [57]:
# Eliminar filas sin fecha valida

data = data.dropna(subset=['dt_ingreso', 'dt_atencion']).copy()

In [58]:
# Calculamos la variable continua de espera en minutos entre la fecha de ingreso y la fecha de atención
data['tiempo_espera_min'] = (data['dt_atencion'] - data['dt_ingreso']).dt.total_seconds() / 60.0

In [59]:
# Agregar mas variables de tiempo para análisis adicionales

data['hora_dia'] = data['dt_ingreso'].dt.hour
data['dia_semana'] = data['dt_ingreso'].dt.day_name()
data['anio_mes'] = data['dt_ingreso'].dt.to_period("M").astype(str)

def asignar_franja(hora):
    if 6 <= hora < 12:
        return "Mañana (06:00 - 11:59)"
    elif 12 <= hora < 18:
        return "Tarde (12:00 - 17:59)"
    elif 18 <= hora < 24:
        return "Noche (18:00 - 23:59)"
    else:
        return "Madrugada (00:00 - 05:59)"

data['franja_horaria'] = data['hora_dia'].apply(asignar_franja)

In [60]:
data.head() # Mostramos las primeras filas del dataset después de los cambios

,CodAdminis,Nom_Admini,Fecha_Ing,Hora_Ingre,Fecha_Atencion,Hora_Atencion,Triage,Ips,Red,dt_ingreso,dt_atencion,tiempo_espera_min,hora_dia,dia_semana,anio_mes,franja_horaria
0,ESS118,EMSSANAR S.A.S,2020 Jan 23 08:29:02 AM,1900 Jan 01 08:29:00 AM,2020 Jan 23 08:36:00 AM,1900 Jan 01 08:36:00 AM,III,HOSPITAL LOCAL CIVIL,RED NORTE,2020-01-23 08:29:02,2020-01-23 08:36:00,6.966667,8,Thursday,2020-01,Mañana (06:00 - 11:59)
2,ESS118,EMSSANAR S.A.S,2020 Jan 14 09:27:46 AM,1900 Jan 01 09:27:00 AM,2020 Jan 14 10:18:00 AM,1900 Jan 01 10:18:00 AM,III,HOSPITAL LOCAL CIVIL,RED NORTE,2020-01-14 09:27:46,2020-01-14 10:18:00,50.233333,9,Tuesday,2020-01,Mañana (06:00 - 11:59)
3,ESS118,EMSSANAR S.A.S,2020 Feb 29 09:01:31 AM,1900 Jan 01 09:01:00 AM,2020 Feb 29 09:11:00 AM,1900 Jan 01 09:11:00 AM,III,CENTRO HOSPITAL LA ROSA,RED SUR,2020-02-29 09:01:31,2020-02-29 09:11:00,9.483333,9,Saturday,2020-02,Mañana (06:00 - 11:59)
4,ESS118,EMSSANAR S.A.S,2020 Feb 09 05:26:00 PM,1900 Jan 01 05:26:00 PM,2020 Feb 09 05:50:00 PM,1900 Jan 01 05:50:00 PM,III,HOSPITAL LOCAL CIVIL,RED NORTE,2020-02-09 17:26:00,2020-02-09 17:50:00,24.000000,17,Sunday,2020-02,Tarde (12:00 - 17:59)
5,ESS118,EMSSANAR S.A.S,2020 Jan 26 04:31:55 PM,1900 Jan 01 04:31:00 PM,2020 Jan 26 04:36:00 PM,1900 Jan 01 04:36:00 PM,III,CS SAN VICENTE,RED OCCIDENTE,2020-01-26 16:31:55,2020-01-26 16:36:00,4.083333,16,Sunday,2020-01,Tarde (12:00 - 17:59)


In [61]:
# Ordenamos cronologicamente y calculamos la variable tiempo entre llegadas

data = data.sort_values(['Ips','dt_ingreso']).reset_index(drop=True)
data['tiempo_entre_llegadas_min'] = (data.groupby('Ips')['dt_ingreso'].diff().dt.total_seconds() /60.0)

In [62]:
# Seleccionamos columnas finales limpias

columnas_finales = [
    "dt_ingreso", "dt_atencion", "anio_mes", "dia_semana", "hora_dia",
    "franja_horaria", "Triage", "Ips", "Red", "Nom_Admini",
    "tiempo_espera_min", "tiempo_entre_llegadas_min"
]

data_limpio = data[columnas_finales]
data_limpio.to_csv("C:\\Users\\soporte.sistemas\\RepoLocal\\Unicafam\\Project_Analitica_II\\SIGU\\urgencias_limpio.csv", index=False)

In [63]:
data_limpio.head()

,dt_ingreso,dt_atencion,anio_mes,dia_semana,hora_dia,franja_horaria,Triage,Ips,Red,Nom_Admini,tiempo_espera_min,tiempo_entre_llegadas_min
0,2020-01-01 00:35:06,2020-01-01 00:38:00,2020-01,Wednesday,0,Madrugada (00:00 - 05:59),II,CENTRO HOSPITAL LA ROSA,RED SUR,EMSSANAR S.A.S,2.900000,NaN
1,2020-01-01 03:28:37,2020-01-01 03:45:00,2020-01,Wednesday,3,Madrugada (00:00 - 05:59),III,CENTRO HOSPITAL LA ROSA,RED SUR,EMSSANAR S.A.S,16.383333,173.516667
2,2020-01-01 04:14:59,2020-01-01 04:38:00,2020-01,Wednesday,4,Madrugada (00:00 - 05:59),III,CENTRO HOSPITAL LA ROSA,RED SUR,COMPAÃÃA MUNDIAL DE SEGUROS S.A.,23.016667,46.366667
3,2020-01-01 04:30:01,2020-01-01 04:47:00,2020-01,Wednesday,4,Madrugada (00:00 - 05:59),III,CENTRO HOSPITAL LA ROSA,RED SUR,EMSSANAR S.A.S,16.983333,15.033333
4,2020-01-01 04:41:38,2020-01-01 05:08:00,2020-01,Wednesday,4,Madrugada (00:00 - 05:59),III,CENTRO HOSPITAL LA ROSA,RED SUR,EMSSANAR S.A.S,26.366667,11.616667


In [64]:
print(f"Registros limpios guardados: {len(data_limpio):,}")
print(f"Tiempo de espera promedio: {data_limpio['tiempo_espera_min'].mean():.2f} min")
print(f"Mediana de espera: {data_limpio['tiempo_espera_min'].median():.2f} min")

Registros limpios guardados: 79,521
Tiempo de espera promedio: 57.02 min
Mediana de espera: 38.60 min


In [65]:
data_limpio.describe()

,dt_ingreso,dt_atencion,hora_dia,tiempo_espera_min,tiempo_entre_llegadas_min
count,79521,79521,79521.000000,79521.000000,79513.000000
mean,2021-11-15 20:06:02.562581,2021-11-15 21:03:03.826159,12.979779,57.021060,82.651718
min,2020-01-01 00:20:32,2020-01-01 00:37:00,0.000000,-0.983333,0.000000
25%,2021-07-23 14:43:57,2021-07-23 15:23:00,9.000000,15.850000,6.000000
50%,2022-01-23 00:30:05,2022-01-23 00:37:00,13.000000,38.600000,13.116667
75%,2022-06-26 08:40:00,2022-06-26 08:50:00,17.000000,83.650000,33.150000
max,2022-09-29 23:45:00,2022-09-30 01:11:00,23.000000,360.000000,724111.616667
std,NaN,NaN,5.313589,53.953481,4968.068411


In [ ]:
import numpy as np

# 1. Eliminar los 172 desfases negativos de reloj en tiempo de espera
data_limpio = data_limpio[data_limpio["tiempo_espera_min"] >= 0].copy()

# 2. Rellenar el único nulo en aseguradora (si las columnas están presentes)
if "CodAdminis" in data_limpio.columns:
    data_limpio["CodAdminis"] = data_limpio["CodAdminis"].fillna("SIN_COD")
if "Nom_Admini" in data_limpio.columns:
    data_limpio["Nom_Admini"] = data_limpio["Nom_Admini"].fillna("NO REPORTADA")

# 3. Recalcular tiempo_entre_llegadas_min agrupando por (Ips, anio_mes)
# Esto evita restar pacientes a través del hueco anual 2020-2021
data_limpio = data_limpio.sort_values(["Ips", "dt_ingreso"]).reset_index(drop=True)
tel_mes = data_limpio.groupby(["Ips", "anio_mes"])["dt_ingreso"].diff().dt.total_seconds() / 60.0

# Descartar saltos mayores a 24 horas (1440 min) y rellenar bordes con la mediana de cada sede (0 nulos)
tel_sin_saltos = tel_mes.where(tel_mes <= 1440, np.nan)
mediana_por_ips = tel_sin_saltos.groupby(data_limpio["Ips"]).transform("median").fillna(tel_sin_saltos.median())

data_limpio["tiempo_entre_llegadas_min"] = tel_sin_saltos.fillna(mediana_por_ips).round(2)
data_limpio["tiempo_espera_min"] = data_limpio["tiempo_espera_min"].round(2)

# 4. Guardar el CSV final listo para backend.py y app.py
data_limpio.to_csv("urgencias_limpio.csv", index=False, encoding="utf-8-sig")

print("Nulos totales:", data_limpio.isnull().sum().sum())
print(data_limpio[["dt_ingreso", "dt_atencion", "hora_dia", "tiempo_espera_min", "tiempo_entre_llegadas_min"]].describe())